# 🫁 Lung Disease — Baseline ResNet50
### Multimodal Architecture — CNN Branch (Baseline)
- **Dataset:** Lungs Disease Dataset (4 types) — Kaggle
- **Model:** ResNet50 (standard — no DSConv, no GCSA)
- **Purpose:** Baseline to compare against the optimized model
- **Metrics:** Accuracy · Recall · F1 Score · Specificity

## ✔️ Cell 0 — Accelerator Check

In [1]:
import torch
print(torch.cuda.is_available())      # Should print: True
print(torch.cuda.get_device_name(0))    # Should print: Tesla T4

True
Tesla T4


## ⚙️ Cell 1 — Configuration
> **Only edit this cell. No other cell needs changes.**
> - `QUICK_TEST = True` → 1/4 data, 5 epochs (verify everything works)
> - `QUICK_TEST = False` → full training

In [2]:
import torch
 
# ── Dataset ─────────────────────────────────────────────────
DATA_ROOT = "/kaggle/input/datasets/omkarmanohardalvi/lungs-disease-dataset-4-types/Lung Disease Dataset"
IMG_SIZE  = 224
 
# ── Quick test mode ──────────────────────────────────────────
QUICK_TEST  = False                    # ← Set False for full training
EPOCHS      = 5  if QUICK_TEST else 30
SUBSET_FRAC = 0.25 if QUICK_TEST else 1.0
 
# ── Training ─────────────────────────────────────────────────
BATCH_SIZE  = 64
LR          = 1e-4
FEATURE_DIM = 512
DROPOUT     = 0.4
NUM_WORKERS = 2           # ← kept at 2 to reduce RAM used by worker processes
PREFETCH    = 2
 
# ── Device & AMP ─────────────────────────────────────────────
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
 
# ── Classes ──────────────────────────────────────────────────
CLASS_NAMES  = ["Bacterial Pneumonia", "Corona Virus Disease",
                "Normal", "Tuberculosis", "Viral Pneumonia"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES  = len(CLASS_NAMES)
 
print(f"Device     : {DEVICE}")
print(f"AMP        : {USE_AMP}")
print(f"Quick Test : {QUICK_TEST}  (subset={SUBSET_FRAC}, epochs={EPOCHS})")
print(f"Batch size : {BATCH_SIZE}")
print(f"Workers    : {NUM_WORKERS}  prefetch={PREFETCH}")

Device     : cuda
AMP        : True
Quick Test : False  (subset=1.0, epochs=30)
Batch size : 64
Workers    : 2  prefetch=2


## 📦 Cell 2 — Imports, Dataset & RAM Utilities
**RAM management strategy:**
- `CachedLungDataset` loads images into RAM for fast epoch iteration
- `free_loader()` explicitly deletes dataloaders + datasets and calls
  `gc.collect()` so Python releases the memory immediately
- GPU cache is cleared with `torch.cuda.empty_cache()` after each phase
- `best_state` (model weight copy) is deleted from RAM right after saving
- `persistent_workers=False` so worker processes exit when loader is deleted

In [3]:
import os, time, copy, gc
from pathlib import Path
 
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset
from torch.amp import GradScaler, autocast
from torchvision import models, transforms
from PIL import Image
 
 
# ── RAM utility ───────────────────────────────────────────────
def free_loader(*loaders):
    """
    Explicitly shut down DataLoader workers and delete all references
    so the OS reclaims the RAM held by the cached dataset immediately.
    Call this as soon as a loader is no longer needed.
    """
    for loader in loaders:
        if loader is not None:
            try:
                loader._iterator = None   # stop prefetch thread if running
            except Exception:
                pass
            del loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
 
def ram_report(label=""):
    """Print current RAM usage for monitoring."""
    import psutil
    used = psutil.virtual_memory().used  / 1e9
    avail= psutil.virtual_memory().available / 1e9
    total= psutil.virtual_memory().total / 1e9
    tag  = f"[{label}] " if label else ""
    print(f"  {tag}RAM — used: {used:.1f}GB / {total:.1f}GB  "
          f"(available: {avail:.1f}GB)")
 
 
# ── Dataset ───────────────────────────────────────────────────
class CachedLungDataset(Dataset):
    """
    Loads all images into RAM once on construction.
    Images stored as PIL grayscale objects — compact in memory.
    Augmentation applied on-the-fly in __getitem__ so only one
    copy of each image sits in RAM (not pre-augmented tensors).
    """
    EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
 
    def __init__(self, root, split="train", img_size=224):
        self.transform = self._get_transforms(split, img_size)
        split_dir      = Path(root) / split
        raw_paths      = []
 
        for cls in CLASS_NAMES:
            d = split_dir / cls
            if not d.exists():
                print(f"  [!] Missing: {d}")
                continue
            for f in d.iterdir():
                if f.suffix.lower() in self.EXT:
                    raw_paths.append((str(f), CLASS_TO_IDX[cls]))
 
        print(f"  [{split.upper():5s}] Loading {len(raw_paths):,} images into RAM...",
              end="", flush=True)
        t0 = time.time()
        self.samples = []
        for path, label in raw_paths:
            img = Image.open(path).convert("L")
            img.load()              # force full pixel decode now
            self.samples.append((img, label))
        print(f" done in {time.time()-t0:.1f}s")
 
    @staticmethod
    def _get_transforms(split, img_size):
        norm = transforms.Normalize([0.485, 0.456, 0.406],
                                    [0.229, 0.224, 0.225])
        if split == "train":
            return transforms.Compose([
                transforms.Grayscale(3),
                transforms.Resize((img_size + 32, img_size + 32)),
                transforms.RandomCrop(img_size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(), norm,
            ])
        return transforms.Compose([
            transforms.Grayscale(3),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(), norm,
        ])
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, i):
        img, label = self.samples[i]
        return self.transform(img), label
 
 
def build_dataloaders(subset_frac=1.0):
    """
    Returns (train_loader, val_loader).
    persistent_workers=False so workers exit when loaders are deleted,
    freeing their RAM immediately.
    """
    train_ds = CachedLungDataset(DATA_ROOT, "train", IMG_SIZE)
    val_ds   = CachedLungDataset(DATA_ROOT, "val",   IMG_SIZE)
 
    if subset_frac < 1.0:
        n_tr     = int(len(train_ds) * subset_frac)
        n_vl     = int(len(val_ds)   * subset_frac)
        train_ds = Subset(train_ds, torch.randperm(len(train_ds))[:n_tr].tolist())
        val_ds   = Subset(val_ds,   torch.randperm(len(val_ds))[:n_vl].tolist())
        print(f"  [SUBSET] train={n_tr}, val={n_vl}")
 
    pin = torch.cuda.is_available()
    kw  = dict(num_workers=NUM_WORKERS, pin_memory=pin,
               prefetch_factor=PREFETCH,
               persistent_workers=False)   # ← False: workers exit on del
 
    return (DataLoader(train_ds, BATCH_SIZE, shuffle=True,  **kw),
            DataLoader(val_ds,   BATCH_SIZE, shuffle=False, **kw))
 
 
def build_test_loader():
    """Test loader — always full dataset, never subset."""
    test_ds = CachedLungDataset(DATA_ROOT, "test", IMG_SIZE)
    pin     = torch.cuda.is_available()
    return DataLoader(test_ds, BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=pin,
                      prefetch_factor=PREFETCH,
                      persistent_workers=False)
 
 
print("Dataset & RAM utilities defined ✓")
ram_report("after imports")

Dataset & RAM utilities defined ✓
  [after imports] RAM — used: 1.2GB / 33.7GB  (available: 32.0GB)


## 🧠 Cell 3 — Baseline ResNet50 Model
**What makes this the BASELINE:**
- Standard ResNet50 pretrained on ImageNet
- Only the final `fc` layer is replaced with a new classification head
- **No** DepthwiseSeparableConv
- **No** GCSA (Global Context Self-Attention)
- **No** layer freezing — all layers train from epoch 1
- `proj` layer maps 2048-d → 512-d for the Feature Fusion module
 
This is intentionally simpler to show the improvement from
the full optimized architecture (DSConv + GCSA + freezing).

In [4]:
class BaselineResNet50(nn.Module):
    """
    Standard ResNet50 with only the fc head replaced.
    No DSConv. No GCSA. No layer freezing.
    extract_features() returns a 512-d vector for Feature Fusion.
    """
    def __init__(self, num_classes, feature_dim=512,
                 pretrained=True, dropout=0.4):
        super().__init__()
 
        w        = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=w)
 
        # Full ResNet50 up to (and including) layer4
        self.features = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,        # output: (B, 2048, H', W')
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)     # (B, 2048, 1, 1)
        self.proj = nn.Linear(2048, feature_dim) # 2048 → 512
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_classes)
        )
 
    def extract_features(self, x):
        """512-d feature vector for the Feature Fusion module."""
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.proj(x)
 
    def forward(self, x):
        return self.head(self.extract_features(x))
 
 
print("Baseline ResNet50 defined ✓")
print("\nDifferences vs optimized model:")
print("  ✗ No DepthwiseSeparableConv")
print("  ✗ No GCSA (Global Context Self-Attention)")
print("  ✗ No layer freezing / two-phase training")
print("  ✓ Same dataset, transforms, AMP, metrics")

Baseline ResNet50 defined ✓

Differences vs optimized model:
  ✗ No DepthwiseSeparableConv
  ✗ No GCSA (Global Context Self-Attention)
  ✗ No layer freezing / two-phase training
  ✓ Same dataset, transforms, AMP, metrics


## 📊 Cell 4 — Metrics & Training Functions
Metrics computed directly from the confusion matrix (no extra libraries):
- **Accuracy** — overall correct predictions
- **Recall** — macro-average true positive rate per class
- **F1 Score** — macro-average harmonic mean of precision and recall
- **Specificity** — macro-average true negative rate per class
 
`te_probs` (softmax probabilities) are **not stored** since graphs are
removed — this alone saves ~20–40 MB of RAM on the test set.

In [5]:
def compute_metrics(y_true, y_pred):
    """
    Accuracy, Macro Recall, Macro F1, Macro Specificity
    from flat lists of true and predicted labels.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
 
    C = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
 
    accuracy = np.trace(C) / C.sum()
    recalls, precisions, specificities, f1s = [], [], [], []
 
    for i in range(NUM_CLASSES):
        tp = C[i, i]
        fn = C[i, :].sum() - tp
        fp = C[:, i].sum() - tp
        tn = C.sum() - tp - fn - fp
 
        recall      = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1          = (2 * precision * recall / (precision + recall)
                       if (precision + recall) > 0 else 0.0)
 
        recalls.append(recall)
        precisions.append(precision)
        specificities.append(specificity)
        f1s.append(f1)
 
    return {
        "accuracy"   : float(accuracy),
        "recall"     : float(np.mean(recalls)),
        "f1"         : float(np.mean(f1s)),
        "specificity": float(np.mean(specificities)),
        "confusion"  : C,
        "per_class"  : {
            CLASS_NAMES[i]: {
                "recall"     : recalls[i],
                "precision"  : precisions[i],
                "f1"         : f1s[i],
                "specificity": specificities[i],
            } for i in range(NUM_CLASSES)
        }
    }
 
 
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum       = 0.
    y_true, y_pred = [], []
 
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
 
        with autocast("cuda", enabled=USE_AMP):
            out  = model(imgs)
            loss = criterion(out, labels)
 
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
 
        loss_sum += loss.item() * len(imgs)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(out.argmax(1).cpu().tolist())
 
    return loss_sum / len(y_true), compute_metrics(y_true, y_pred)
 
 
@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    """
    Returns loss and metrics.
    Does NOT store softmax probs (graphs removed) — saves RAM.
    """
    model.eval()
    loss_sum       = 0.
    y_true, y_pred = [], []
 
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=USE_AMP):
            out  = model(imgs)
            loss = criterion(out, labels)
 
        loss_sum += loss.item() * len(imgs)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(out.argmax(1).cpu().tolist())
 
    return loss_sum / len(y_true), compute_metrics(y_true, y_pred)
 
 
print("Metrics & training functions defined ✓")

Metrics & training functions defined ✓


## 🚀 Cell 5 — Train Baseline ResNet50
**RAM management in this cell:**
1. Train & val loaders built → used for training → deleted immediately after
2. Test loader built separately → evaluated → deleted immediately after
3. `best_state` (copy of best weights) deleted from RAM after `torch.save()`
4. `torch.cuda.empty_cache()` called after training and after test evaluation
5. Final RAM report printed so you can monitor usage

In [6]:
ram_report("before training")
 
print(f"\n{'='*65}")
print(f"  Training : BASELINE ResNet50")
print(f"  Mode     : {'QUICK TEST (1/4 data)' if QUICK_TEST else 'FULL TRAINING'}")
print(f"  AMP      : {USE_AMP}   Layer freezing: NONE")
print(f"{'='*65}")
 
# ── Build train & val loaders ─────────────────────────────
train_loader, val_loader = build_dataloaders(SUBSET_FRAC)
ram_report("after loading train+val into RAM")
 
# ── Model ─────────────────────────────────────────────────
model    = BaselineResNet50(NUM_CLASSES, FEATURE_DIM,
                            pretrained=True, dropout=DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\n  Total parameters    : {n_params:.2f}M  (all trainable, no freezing)\n")
 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler("cuda", enabled=USE_AMP)
 
best_val_acc, best_state = 0.0, None
history = {k: [] for k in ["train_loss", "val_loss",
                            "train_acc",  "val_acc",
                            "train_f1",   "val_f1"]}
 
print(f"  {'Ep':>3} {'Tr Loss':>9} {'Tr Acc':>7} {'Tr F1':>6} "
      f"{'Val Loss':>9} {'Val Acc':>8} {'Val F1':>7} {'Time':>6}")
print("  " + "─" * 62)
 
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_met = train_epoch(model, train_loader,
                                  criterion, optimizer, scaler)
    vl_loss, vl_met = evaluate_loader(model, val_loader, criterion)
    scheduler.step()
 
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_met["accuracy"])
    history["val_acc"].append(vl_met["accuracy"])
    history["train_f1"].append(tr_met["f1"])
    history["val_f1"].append(vl_met["f1"])
 
    flag = " ✓" if vl_met["accuracy"] > best_val_acc else ""
    print(f"  {epoch:3d} {tr_loss:9.4f} {tr_met['accuracy']:7.4f} "
          f"{tr_met['f1']:6.4f} {vl_loss:9.4f} {vl_met['accuracy']:8.4f} "
          f"{vl_met['f1']:7.4f} {time.time()-t0:5.1f}s{flag}")
 
    if vl_met["accuracy"] > best_val_acc:
        best_val_acc = vl_met["accuracy"]
        best_state   = copy.deepcopy(model.state_dict())
 
# ── Free train & val loaders immediately ──────────────────
free_loader(train_loader, val_loader)
del train_loader, val_loader
ram_report("after freeing train+val loaders")
 
# ── Load best weights ─────────────────────────────────────
model.load_state_dict(best_state)
 
# ── Save checkpoint ───────────────────────────────────────
SAVE_PATH = "/kaggle/working/baseline_resnet50_best.pth"
torch.save({
    "model"       : "baseline_resnet50",
    "state_dict"  : best_state,
    "val_acc"     : best_val_acc,
    "class_names" : CLASS_NAMES,
    "feature_dim" : FEATURE_DIM,
    "quick_test"  : QUICK_TEST,
}, SAVE_PATH)
 
# ── Free best_state from RAM immediately after saving ─────
del best_state
gc.collect()
print(f"\n  ✓ Checkpoint saved → {SAVE_PATH}")
 
# ── Build test loader & evaluate ──────────────────────────
print("\n  Running final evaluation on test set...")
test_loader = build_test_loader()
ram_report("after loading test set into RAM")
 
te_loss, te_met = evaluate_loader(model, test_loader, criterion)
 
# ── Free test loader immediately ──────────────────────────
free_loader(test_loader)
del test_loader
torch.cuda.empty_cache()
gc.collect()
ram_report("after freeing test loader")
 
# ── Results summary ───────────────────────────────────────
print(f"\n  {'═'*52}")
print(f"  {'BASELINE ResNet50 — FINAL RESULTS':^52}")
print(f"  {'═'*52}")
print(f"  {'Metric':<16} {'Val (best epoch)':>18}  {'Test Set':>10}")
print(f"  {'─'*52}")
print(f"  {'Accuracy':<16} {best_val_acc*100:>17.2f}%  {te_met['accuracy']*100:>9.2f}%")
print(f"  {'Recall':<16} {'—':>18}  {te_met['recall']*100:>9.2f}%")
print(f"  {'F1 Score':<16} {'—':>18}  {te_met['f1']*100:>9.2f}%")
print(f"  {'Specificity':<16} {'—':>18}  {te_met['specificity']*100:>9.2f}%")
print(f"  {'─'*52}")
print(f"\n  Per-class breakdown (Test Set):")
print(f"  {'Class':<24} {'Recall':>7} {'Precision':>10} {'F1':>7} {'Spec':>8}")
print(f"  {'─'*60}")
for cls in CLASS_NAMES:
    m = te_met["per_class"][cls]
    print(f"  {cls:<24} {m['recall']*100:>6.2f}%  {m['precision']*100:>8.2f}%  "
          f"{m['f1']*100:>6.2f}%  {m['specificity']*100:>7.2f}%")
print(f"  {'─'*60}")

  [before training] RAM — used: 1.2GB / 33.7GB  (available: 32.0GB)

  Training : BASELINE ResNet50
  Mode     : FULL TRAINING
  AMP      : True   Layer freezing: NONE
  [TRAIN] Loading 6,054 images into RAM... done in 97.2s
  [VAL  ] Loading 2,016 images into RAM... done in 32.8s
  [after loading train+val into RAM] RAM — used: 10.6GB / 33.7GB  (available: 22.6GB)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s]



  Total parameters    : 24.56M  (all trainable, no freezing)

   Ep   Tr Loss  Tr Acc  Tr F1  Val Loss  Val Acc  Val F1   Time
  ──────────────────────────────────────────────────────────────
    1    0.7913  0.7922 0.7885    0.6585   0.8641  0.8609  58.1s ✓
    2    0.6350  0.8763 0.8751    0.6316   0.8755  0.8687  56.4s ✓
    3    0.5990  0.8981 0.8973    0.6525   0.8666  0.8625  56.5s
    4    0.5754  0.9068 0.9062    0.7027   0.8512  0.8476  57.2s
    5    0.5585  0.9204 0.9200    0.6928   0.8606  0.8573  56.9s
    6    0.5421  0.9319 0.9316    0.6880   0.8805  0.8747  57.0s ✓
    7    0.5285  0.9362 0.9358    0.5934   0.9107  0.9096  57.8s ✓
    8    0.5104  0.9480 0.9477    0.6514   0.8730  0.8686  57.2s
    9    0.4995  0.9519 0.9517    0.6844   0.8552  0.8427  56.6s
   10    0.4856  0.9575 0.9573    0.7029   0.8462  0.8323  57.1s
   11    0.4712  0.9673 0.9671    0.6321   0.8765  0.8684  56.7s
   12    0.4609  0.9701 0.9699    0.5949   0.9082  0.9065  56.8s
   13    0.4448  0.

## 📤 Cell 6 — Extract Feature Vectors (512-d)
Loads the saved best checkpoint, runs inference on the val set,
and saves 512-d feature vectors for the Feature Fusion module.
 
**RAM management:** A fresh model instance is created, used,
then deleted along with its loader once extraction is complete.

In [7]:
ram_report("before feature extraction")
 
# ── Build val loader fresh (train loader was already freed) ─
_, val_loader_feat = build_dataloaders(SUBSET_FRAC)
 
# ── Load best checkpoint into a fresh model instance ────────
model_feat = BaselineResNet50(NUM_CLASSES, FEATURE_DIM,
                              pretrained=False).to(DEVICE)
ckpt = torch.load(SAVE_PATH, weights_only=False)
model_feat.load_state_dict(ckpt["state_dict"])
model_feat.eval()
del ckpt
gc.collect()
 
# ── Extract features batch by batch ─────────────────────────
all_features, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader_feat:
        feats = model_feat.extract_features(imgs.to(DEVICE))
        all_features.append(feats.cpu())
        all_labels.append(labels)
 
all_features = torch.cat(all_features, dim=0)    # (N, 512)
all_labels   = torch.cat(all_labels,   dim=0)    # (N,)
 
# ── Save ────────────────────────────────────────────────────
feat_path = "/kaggle/working/baseline_resnet50_features.pt"
torch.save({
    "features"   : all_features,
    "labels"     : all_labels,
    "class_names": CLASS_NAMES,
    "model"      : "baseline_resnet50",
}, feat_path)
 
print(f"Feature shape : {tuple(all_features.shape)}")
print(f"Saved         → {feat_path}")
 
# ── Free everything ─────────────────────────────────────────
free_loader(val_loader_feat)
del val_loader_feat, model_feat, all_features, all_labels
gc.collect()
torch.cuda.empty_cache()
ram_report("after feature extraction — all freed")
 
# ── Final file listing ───────────────────────────────────────
print(f"\n{'='*55}")
print("  Output files in /kaggle/working/:")
print(f"{'='*55}")
for fname in sorted(os.listdir("/kaggle/working")):
    fpath = f"/kaggle/working/{fname}"
    if os.path.isfile(fpath) and not fname.endswith(".db"):
        size = os.path.getsize(fpath) / 1e6
        print(f"  {fname:<45} {size:.1f} MB")

  [before feature extraction] RAM — used: 11.4GB / 33.7GB  (available: 21.5GB)
  [TRAIN] Loading 6,054 images into RAM... done in 99.0s
  [VAL  ] Loading 2,016 images into RAM... done in 29.1s
Feature shape : (2016, 512)
Saved         → /kaggle/working/baseline_resnet50_features.pt
  [after feature extraction — all freed] RAM — used: 11.6GB / 33.7GB  (available: 21.2GB)

  Output files in /kaggle/working/:
  __notebook__.ipynb                            0.0 MB
  baseline_resnet50_best.pth                    98.6 MB
  baseline_resnet50_features.pt                 4.1 MB
